# Notebook 01 — Prepare Augmented Training Data

Produces `train_dm.csv` and `valid_dm.csv` (tab-delimited, columns: `file`, `label`, `path`)  
consumed by notebook 02 for fine-tuning.

**Pipeline**
1. Scan original audio directories → build file-level DataFrames
2. Stratified 80/20 train/validation split on originals only
3. Apply SpecAugment (Park et al. 2019) to training originals only
4. Save `train_dm.csv` (originals + augmented) and `valid_dm.csv` (originals only)

In [ ]:
import os
import json
import shutil

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import torchaudio
import IPython.display

from pathlib import Path
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
save_path = Path('/content/drive/MyDrive/CS7357/Project/data')

dm_path = save_path / 'audio' / 'dementia'
nd_path = save_path / 'audio' / 'nodementia'
aug_dir = save_path / 'augmented'
if aug_dir.exists():
    shutil.rmtree(aug_dir)
aug_dir.mkdir()

print(f'Dementia audio  : {dm_path}')
print(f'Nodementia audio: {nd_path}')
print(f'Augmented output: {aug_dir}')

In [ ]:
# Audio player helper — workaround for VS Code / Colab compatibility
def Audio(audio: np.ndarray, sr: int):
    if np.ndim(audio) == 1:
        channels = [audio.tolist()]
    else:
        channels = audio.tolist()
    return IPython.display.HTML("""
        <script>
            if (!window.audioContext) {
                window.audioContext = new AudioContext();
                window.playAudio = function(audioChannels, sr) {
                    const buffer = audioContext.createBuffer(audioChannels.length, audioChannels[0].length, sr);
                    for (let [channel, data] of audioChannels.entries()) {
                        buffer.copyToChannel(Float32Array.from(data), channel);
                    }
                    const source = audioContext.createBufferSource();
                    source.buffer = buffer;
                    source.connect(audioContext.destination);
                    source.start();
                }
            }
        </script>
        <button onclick=\"playAudio(%s, %s)\">Play</button>
    """ % (json.dumps(channels), sr))

## 1. Scan Original Audio Directories

In [ ]:
def scan_wavs(base_path: Path, label: str) -> list:
    """Walk base_path recursively and return one dict per .wav file."""
    rows = []
    for root, _, files in os.walk(base_path):
        for f in files:
            if f.lower().endswith('.wav'):
                p = Path(root) / f
                rows.append({'file': p.stem, 'label': label, 'path': str(p)})
    return rows


dm_df = pd.DataFrame(scan_wavs(dm_path, 'dementia'))
nd_df = pd.DataFrame(scan_wavs(nd_path, 'nodementia'))

print(f'Dementia files  : {len(dm_df)}')
print(f'Nodementia files: {len(nd_df)}')

if dm_df.empty or nd_df.empty:
    raise RuntimeError(
        'One or both audio directories returned 0 files. '
        'Check that the paths above exist and the Drive is mounted.'
    )

## 2. Stratified Train / Validation Split (80 / 20)

The split is performed on the **original** recordings only.  
Augmented copies will be added to the training set afterward — they never touch validation.

In [ ]:
full_df = pd.concat([dm_df, nd_df], ignore_index=True)

train_df, valid_df = train_test_split(
    full_df,
    test_size=0.2,
    stratify=full_df['label'],
    random_state=42,
)
train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

# Sanity check — no file should appear in both splits
overlap = set(train_df['file']) & set(valid_df['file'])
assert len(overlap) == 0, f'Data leakage detected: {overlap}'

print(f'Training originals  : {len(train_df)}')
print(train_df['label'].value_counts().to_string())
print()
print(f'Validation originals: {len(valid_df)}')
print(valid_df['label'].value_counts().to_string())
print()
print('No overlap between train and validation sets. OK')

## 3. SpecAugment Configuration

**SM (Switchboard Mild)** policy from Park et al. (2019).

| Param | Value | Meaning |
|-------|-------|---------|
| F     | 15    | Max frequency mask width (STFT bins) |
| mF    | 2     | Number of frequency masks |
| T     | 70    | Max time mask width (STFT frames) |
| p     | 0.2   | Max proportion of time frames masked |
| mT    | 2     | Number of time masks |
| copies| 2     | Augmented copies per original training sample |

In [ ]:
SPEC_CONFIG = {
    'n_fft':       1024,  # FFT window size
    'hop_length':   256,  # STFT hop length
    'F':             15,  # Max frequency mask width (bins)
    'mF':             2,  # Number of frequency masks
    'T':             70,  # Max time mask width (frames)
    'p':            0.2,  # Max fraction of frames that can be masked
    'mT':             2,  # Number of time masks
    'num_augments': {'dementia': 3, 'nodementia': 2},  # more copies for the minority class
}

print('SpecAugment — SM (Switchboard Mild) policy')
print(f"  Frequency mask : F={SPEC_CONFIG['F']}, applied {SPEC_CONFIG['mF']}x")
print(f"  Time mask      : T={SPEC_CONFIG['T']}, p={SPEC_CONFIG['p']}, applied {SPEC_CONFIG['mT']}x")
print(f"  Copies / sample: dementia={SPEC_CONFIG['num_augments']['dementia']}, "
      f"nodementia={SPEC_CONFIG['num_augments']['nodementia']}")

In [ ]:
def spec_augment_wav(waveform: torch.Tensor, sr: int, config: dict) -> torch.Tensor:
    """
    Apply SpecAugment-style masking (Park et al. 2019) in the STFT domain.

    Pipeline:
      1. STFT  -> complex spectrogram
      2. Apply frequency masking (mF times)
      3. Apply time masking     (mT times, capped at p * tau frames)
      4. iSTFT -> waveform (same length as input)

    The result can be saved as a standard .wav with torchaudio.save() and
    loaded in notebook 02 with torchaudio.load().

    Args:
        waveform : (C, T) tensor from torchaudio.load()
        sr       : sample rate (passed through for caller convenience)
        config   : SPEC_CONFIG dict

    Returns:
        (1, T) mono waveform tensor
    """
    n_fft = config['n_fft']
    hop   = config['hop_length']

    # Collapse to mono
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    window    = torch.hann_window(n_fft)
    stft      = torch.stft(waveform[0], n_fft=n_fft, hop_length=hop,
                           window=window, return_complex=True)
    magnitude = stft.abs()
    phase     = stft.angle()
    n_freq, tau = magnitude.shape

    # Frequency masking
    for _ in range(config['mF']):
        f  = torch.randint(1, config['F'] + 1, (1,)).item()
        f0 = torch.randint(0, max(1, n_freq - f), (1,)).item()
        magnitude[f0 : f0 + f, :] = 0.0

    # Time masking
    adaptive_T = min(config['T'], int(config['p'] * tau))
    if adaptive_T > 0:
        for _ in range(config['mT']):
            t  = torch.randint(1, adaptive_T + 1, (1,)).item()
            t0 = torch.randint(0, max(1, tau - t), (1,)).item()
            magnitude[:, t0 : t0 + t] = 0.0

    # Reconstruct and invert
    masked_stft = magnitude * torch.exp(1j * phase)
    augmented   = torch.istft(masked_stft, n_fft=n_fft, hop_length=hop,
                              window=window, length=waveform.shape[1])
    return augmented.unsqueeze(0)  # (1, samples)

## 4. Visualize SpecAugment Effect

Sanity check on one training sample before running the full augmentation loop.

In [ ]:
sample_row = train_df.iloc[0]
waveform, sr = torchaudio.load(sample_row['path'])

n_panels = 3  # original + 2 augmented previews
fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 4))

def plot_spec(ax, wav, title):
    spec = torch.stft(
        wav[0],
        n_fft=SPEC_CONFIG['n_fft'],
        hop_length=SPEC_CONFIG['hop_length'],
        window=torch.hann_window(SPEC_CONFIG['n_fft']),
        return_complex=True,
    )
    db = 20 * torch.log10(spec.abs() + 1e-9)
    ax.imshow(db.numpy(), aspect='auto', origin='lower', cmap='viridis')
    ax.set_title(title)
    ax.set_xlabel('Time Frame')
    ax.set_ylabel('Freq Bin')

plot_spec(axes[0], waveform, 'Original')
for i in range(n_panels - 1):
    aug = spec_augment_wav(waveform, sr, SPEC_CONFIG)
    plot_spec(axes[i + 1], aug, f'Augmented #{i + 1}')

plt.suptitle(f"SpecAugment — {sample_row['file']}", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Augmentation Loop — Training Originals Only

`train_df` contains exclusively original recordings from the directory scan above.  
Each original produces exactly `num_augments` augmented `.wav` files.  
The validation set is never touched.

In [ ]:
augmented_rows = []
skipped        = []

for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc='Augmenting originals'):
    try:
        waveform, sr = torchaudio.load(row['path'])

        n_aug = SPEC_CONFIG['num_augments'][row['label']]
        for aug_i in range(n_aug):
            aug_wav  = spec_augment_wav(waveform, sr, SPEC_CONFIG)
            aug_name = f"{row['file']}_aug{aug_i}"
            aug_file = aug_dir / f'{aug_name}.wav'

            torchaudio.save(str(aug_file), aug_wav, sr)

            augmented_rows.append({
                'file':  aug_name,
                'label': row['label'],
                'path':  str(aug_file),
            })

    except Exception as e:
        skipped.append((row['path'], str(e)))

print(f"Augmented files created: {len(augmented_rows)}")
for lbl, n in SPEC_CONFIG['num_augments'].items():
    count = len(train_df[train_df['label'] == lbl])
    print(f'  {lbl}: {count} originals x {n} copies = {count * n} augmented')
if skipped:
    print(f'Skipped due to errors : {len(skipped)}')
    for p, e in skipped[:5]:
        print(f'  {p}: {e}')

## 6. Build Final Splits and Save CSVs

- `train_dm.csv` — originals + augmented (training in notebook 02)
- `valid_dm.csv` — original recordings only (evaluation in notebook 02)

Both are **tab-delimited** to match notebook 02's `load_dataset(..., delimiter='\t')` call.

In [ ]:
aug_df = pd.DataFrame(augmented_rows)

# Training set: originals + augmented copies
final_train_df = pd.concat([train_df, aug_df], ignore_index=True)

print('=== Final Dataset Summary ===')
print(f'Train — originals : {len(train_df)}')
print(f'Train — augmented : {len(aug_df)}')
print(f'Train — total     : {len(final_train_df)}')
print()
print('Training label distribution:')
print(final_train_df['label'].value_counts().to_string())
print()
print(f'Validation — total: {len(valid_df)}')
print('Validation label distribution:')
print(valid_df['label'].value_counts().to_string())

# Save — tab-delimited, no index
final_train_df.to_csv(save_path / 'train_dm.csv', sep='\t', index=False)
valid_df.to_csv(      save_path / 'valid_dm.csv', sep='\t', index=False)

print(f'\nSaved train_dm.csv : {len(final_train_df)} rows -> {save_path / "train_dm.csv"}')
print(f'Saved valid_dm.csv : {len(valid_df)} rows -> {save_path / "valid_dm.csv"}')